RGPD - Identification et pseudonymisation des données personnelles
=> identifier les colonnes à caractere personnel et appliquer une technique de pseudonymisation conforme au RGPD ( hashing SHA-256 + salt)

In [1]:
import pandas as pd
df = pd.read_csv('../data/raw/Sample - Superstore.csv')

In [4]:
for col in df.columns:
    print(f"{col:20s} | {df[col].nunique():6d} uniques")

Row ID               |   9994 uniques
Order ID             |   5009 uniques
Order Date           |   1495 uniques
Ship Date            |   1334 uniques
Ship Mode            |      4 uniques
Customer ID          |    793 uniques
Customer Name        |    810 uniques
Segment              |      6 uniques
Country              |      1 uniques
City                 |    633 uniques
State                |     86 uniques
Postal Code          |    705 uniques
Region               |      4 uniques
Product ID           |   1862 uniques
Category             |      6 uniques
Sub-Category         |     17 uniques
Product Name         |   1850 uniques
Sales                |   5738 uniques
Quantity             |     15 uniques
Discount             |     13 uniques
Profit               |   7287 uniques


Non PII : ROW ID, Order ID, Order Date, Ship Date,
Ship Mode, Segment, Country, State, Regiopn, Product ID, Category, Sales, Quantity, Discount, Profit) => le garder comme elles
PII indirect: Customer ID, City, Postal Code => le garder aussi
PII direct :  Customer Name => pseudonymiser par hashing SHA-256 avec salt.

La pseudonymisation SHA-256 salée est une méthode cryptographique qui utilise un sel secret avec HMAC-SHA256 pour transformer de manière irréversible des identifiants sensibles en pseudonymes uniques.
Il préserve la structure des données dans des formats tels que JSON et XML tout en prenant en charge la réversibilité contrôlée et les pistes d'audit pour la conformité.
Cette approche protège contre les attaques par force brute, par liaison et quantiques, offrant une forte résistance aux collisions et une protection robuste de la vie privée.

In [6]:
import hashlib
import hmac

def pseudonymize_row(row, salt_key):
    # Concaténation de la valeur et du sel
    data = f"{row}{salt_key}".encode('utf-8')
    # Calcul du HMAC-SHA256
    hashed = hmac.new(salt_key.encode('utf-8'), data, hashlib.sha256).hexdigest()
    return hashed

# Application au DataFrame
df['Customer Name'] = df['Customer Name'].apply(lambda x: pseudonymize_row(x, 'SECRET_SALT_KEY'))
df['Customer Name']

0        21b5b7c44749bfa1495eed34c4779134ff31494e1957cd...
1        21b5b7c44749bfa1495eed34c4779134ff31494e1957cd...
2        7ffcc35fb1e9df2315b2b1a75c146697860a79627e08e5...
3        140e9ee473e56c04db7eb0f13a818af4d9a1a63161e3ed...
4        140e9ee473e56c04db7eb0f13a818af4d9a1a63161e3ed...
                               ...                        
10059    fc09c44bf1b987867a27f3c1330bb27f291e4313aadd6a...
10060    39de06295c0eb9a3611c833c702b8232cc7847a193ee31...
10061    f268a86cc4e781c45398e7a2e7404c7e3d0c27202373c6...
10062    65ac08f7b52e77dc6b89d878d39717132bdb77137e2661...
10063    178e641a2efdc78169fab507404b6e3d23f927fa663b9b...
Name: Customer Name, Length: 10064, dtype: object

In [7]:
df

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,21b5b7c44749bfa1495eed34c4779134ff31494e1957cd...,Consumer,United States,Henderson,...,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2.0,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,21b5b7c44749bfa1495eed34c4779134ff31494e1957cd...,Consumer,United States,Henderson,...,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3.0,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,7ffcc35fb1e9df2315b2b1a75c146697860a79627e08e5...,Corporate,United States,Los Angeles,...,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2.0,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,140e9ee473e56c04db7eb0f13a818af4d9a1a63161e3ed...,Consumer,United States,Fort Lauderdale,...,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5.0,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,140e9ee473e56c04db7eb0f13a818af4d9a1a63161e3ed...,Consumer,United States,Fort Lauderdale,...,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2.0,0.20,2.5164
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10059,1281,CA-2016-160815,9/5/2016,9/6/2016,First Class,TR-21325,fc09c44bf1b987867a27f3c1330bb27f291e4313aadd6a...,Consumer,United States,Cedar Rapids,...,52402.0,Central,TEC-PH-10003505,Technology,Phones,Geemarc AmpliPOWER60,278.4000,3.0,0.00,80.7360
10060,7699,CA-2017-151799,12/14/2017,12/18/2017,Standard Class,BF-11170,39de06295c0eb9a3611c833c702b8232cc7847a193ee31...,Home Office,United States,Lawrence,...,1841.0,East,OFF-ST-10002790,Office Supplies,Storage,Safco Industrial Shelving,73.8500,1.0,0.00,2.2155
10061,8597,CA-2014-111934,5/5/2014,5/7/2014,First Class,GD-14590,f268a86cc4e781c45398e7a2e7404c7e3d0c27202373c6...,Corporate,United States,Arlington,...,22204.0,South,OFF-PA-10000474,Office Supplies,Paper,Easy-staple paper,35.4400,1.0,0.00,16.6568
10062,4962,CA-2014-156587,3/7/2014,3/8/2014,First Class,AB-10015,65ac08f7b52e77dc6b89d878d39717132bdb77137e2661...,Consumer,United States,Seattle,...,98103.0,West,FUR-CH-10004477,Furniture,Chairs,"Global Push Button Manager's Chair, Indigo",48.7120,1.0,0.20,5.4801
